## Change of coordinates

In [ ]:
import sys
from pathlib import Path

import folium
import matplotlib.pyplot as plt

project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root))  

from config import Color  # noqa: E402
from helpers.coordinates import ENUPose, GRAPose  # noqa: E402
from plan.planner import Plan  # noqa: E402


In [ ]:
homes = ENUPose.list([  # east, north, up, heading
    #(0., 0., 0., 0),
    (20., 0., 0., 20),
    (0, -20., 0., 0),
    # (-15., 0., 0., 0.),
    # (0., -20., 0., 0.),
])

paths = [
    Plan.create_square_path(
        side_len=10, alt=5, heading=0, clockwise=True
    ) for _ in homes
]


h_paths = [home.to_abs_all(path) for home,path in zip(homes,paths)]
colors=[
    Color.BLUE.value,
    Color.GREEN.value,
    # Color.YELLOW.value,
    # Color.ORANGE.value,
    # Color.RED.value,
]

## ENU

In [ ]:
origin = ENUPose(30, 0, 0, 45)
o_paths = [origin.to_abs_all(path) for path in h_paths]

In [ ]:
fig, ax = plt.subplots()# type: ignore
ax.set_aspect("equal")
ax.grid(True)# type: ignore
origin.draw(ax, "O", "black")
ENUPose(0,0,0,0).draw(ax, "h", "black",alpha=0.5)
for h_path,o_path,color in zip(h_paths,o_paths,colors):
    for i,(h_pos,o_pos) in enumerate(zip(h_path,o_path)):
        h_pos.draw(ax, f"h_{i}", color,alpha=0.5)  # plotted relative to origin, visually shifted
        o_pos.draw(ax, f"o_{i}", color)
plt.title("Pose transformation in ENU frame")# type: ignore
plt.show()# type: ignore

## GRA 

In [ ]:
origin = GRAPose(-35.3633245, 149.1652241, 0, 45)
o_paths = [origin.to_abs_all(path) for path in h_paths]
colors = [Color.BLUE, Color.GREEN]
origin_color = Color.BLACK
lat0, lon0 = origin[:2]
m = folium.Map(location=[lat0, lon0], zoom_start=18)

# Plot each UAV's path
for path, color in zip(o_paths, colors):  # add more colors if needed
    for i, wp in enumerate(path):
        wp.unpose().draw(m, f"pos_{i}", color)

# Plot origin
origin.unpose().draw(m, "Origin", origin_color)
display(m)
